# Chapter 5: Higher-Dimensional Kernels

This notebook contains every executable artifact used by Chapter 5. The cells move from shapes and strides through matrix addition, transpose, scalar-output matrix multiplication, aligned tiled kernels, the final boundary-safe autotuned kernel, correctness testing, benchmarking, the recorded H200 results, and the roofline calculation.

Run the cells in order on a GPU system with PyTorch and Triton installed. The two intermediate tiled kernels deliberately retain an aligned-shape contract. The final kernel accepts rectangular matrices and dimensions with remainders. No cell writes a CSV, JSON file, archive, image, or other companion artifact.

## Environment and imports

The notebook uses NumPy and Numba for the CPU memory-order experiment and PyTorch and Triton for the GPU kernels. The cells contain no saved outputs, which keeps results tied to the machine on which you execute them.

In [ ]:
import platform
import statistics
import time

import numpy as np
import torch
import triton
import triton.language as tl
from numba import njit


def high_precision_reference(a, b):
    previous_precision = torch.get_float32_matmul_precision()
    try:
        torch.set_float32_matmul_precision("highest")
        reference = torch.matmul(a.float(), b.float())
    finally:
        torch.set_float32_matmul_precision(previous_precision)
    return reference.to(dtype=a.dtype)

## Memory order, strides, and views

A row-major traversal consumes adjacent elements before advancing to the next row. A column traversal performs the same arithmetic in a different address order, so the timing difference comes from locality rather than the number of additions. The functions use compiled loops to keep Python interpreter overhead outside the measurement.

In [ ]:
import numpy as np
import time
from numba import njit

N = 4096
RUNS = 7


@njit(cache=True)
def row_major_sum(matrix):
    rows, cols = matrix.shape
    acc = np.float64(0)
    for row in range(rows):
        for column in range(cols):
            acc += matrix[row, column]
    return acc


@njit(cache=True)
def column_major_sum(matrix):
    rows, cols = matrix.shape
    acc = np.float64(0)
    for column in range(cols):
        for row in range(rows):
            acc += matrix[row, column]
    return acc


rng = np.random.default_rng(0)
cpu_matrix = rng.random((N, N), dtype=np.float32)

row_major_sum(cpu_matrix)
column_major_sum(cpu_matrix)

row_times = []
for _ in range(RUNS):
    start = time.perf_counter()
    row_major_sum(cpu_matrix)
    row_times.append(time.perf_counter() - start)

column_times = []
for _ in range(RUNS):
    start = time.perf_counter()
    column_major_sum(cpu_matrix)
    column_times.append(time.perf_counter() - start)

row_best = min(row_times)
column_best = min(column_times)

print(f"Row-major time : {row_best:.6f} sec")
print(f"Column time    : {column_best:.6f} sec")
print(f"Slowdown       : {column_best / row_best:.2f}x")

PyTorch records a stride for each tensor dimension. Transposing a matrix exchanges the two strides while preserving the underlying storage, and expanding a vector introduces a zero stride along the broadcast dimension. The following checks expose those mappings directly.

In [ ]:
import torch
A = torch.arange(12, dtype=torch.float32).reshape(3, 4)
print(A)
print(A.stride())

transposed = A.t()
bias = torch.arange(4, dtype=torch.float32)
broadcast_bias = bias.expand(3, 4)

print("transpose shape and strides:", transposed.shape, transposed.stride())
print("broadcast shape and strides:", broadcast_bias.shape, broadcast_bias.stride())
print("A and transpose share storage:", A.data_ptr() == transposed.data_ptr())

assert A.stride() == (4, 1)
assert transposed.stride() == (1, 4)
assert broadcast_bias.stride() == (0, 1)
assert A.data_ptr() == transposed.data_ptr()

## Matrix addition with a two-dimensional launch grid

Each Triton program owns one rectangular output tile. The program builds separate row and column offsets, broadcasts them into a two-dimensional pointer grid, and applies one boundary mask to both loads and the store. The host wrapper launches enough programs to cover matrices whose dimensions do not divide evenly by the tile sizes.

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def matrix_add_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N,
    stride_am, stride_an,
    stride_bm, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)

    row_start = pid_m * BLOCK_M
    col_start = pid_n * BLOCK_N

    row_offs = row_start + tl.arange(0, BLOCK_M)
    col_offs = col_start + tl.arange(0, BLOCK_N)

    mask = (
        (tl.expand_dims(row_offs, 1) < M)
        & (tl.expand_dims(col_offs, 0) < N)
    )

    a_ptrs = (
        a_ptr
        + tl.expand_dims(row_offs, 1) * stride_am
        + tl.expand_dims(col_offs, 0) * stride_an
    )
    b_ptrs = b_ptr + tl.expand_dims(row_offs, 1) * stride_bm \
                   + tl.expand_dims(col_offs, 0) * stride_bn

    a = tl.load(a_ptrs, mask=mask, other=0.0)
    b = tl.load(b_ptrs, mask=mask, other=0.0)
    c = a + b

    c_ptrs = c_ptr + tl.expand_dims(row_offs, 1) * stride_cm \
                   + tl.expand_dims(col_offs, 0) * stride_cn
    tl.store(c_ptrs, c, mask=mask)


def matrix_add(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    assert a.shape == b.shape
    M, N = a.shape
    c = torch.empty_like(a)
    BLOCK_M, BLOCK_N = 32, 32
    grid = (
        triton.cdiv(M, BLOCK_M),
        triton.cdiv(N, BLOCK_N),
    )
    matrix_add_kernel[grid](
        a, b, c,
        M, N,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N,
    )
    return c

In [ ]:
device = triton.runtime.driver.active.get_active_torch_device()
torch.manual_seed(0)

add_a = torch.randn((1025, 1537), device=device, dtype=torch.float32)
add_b = torch.randn_like(add_a)
add_result = matrix_add(add_a, add_b)

torch.testing.assert_close(add_result, add_a + add_b, rtol=0, atol=0)
print("Matrix addition matches PyTorch.")

## Naive and tiled transpose

The naive kernel reads a row-major tile through contiguous column offsets and writes the values with the output indices exchanged. Those exchanged indices make neighboring lanes write a full output-row stride apart. The block-pointer kernel describes the source and destination tiles explicitly, transposes the loaded block value, and lets Triton's compiler lower the resulting layout conversion for the active target.

In [ ]:
def transpose_with_loops(x):
    M, N = x.shape
    out = torch.empty((N, M), device=x.device, dtype=x.dtype)
    for row in range(M):
        for col in range(N):
            out[col, row] = x[row, col]
    return out


import triton
import triton.language as tl

@triton.jit
def transpose_naive_kernel(x_ptr, y_ptr, M, N,
                           stride_xm, stride_xn, stride_ym, stride_yn,
                           BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)

    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)[:, None]
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)[None, :]

    mask = (rows < M) & (cols < N)

    x_ptrs = x_ptr + rows * stride_xm + cols * stride_xn
    tile = tl.load(x_ptrs, mask=mask, other=0.0)

    y_ptrs = y_ptr + cols * stride_ym + rows * stride_yn
    tl.store(y_ptrs, tile, mask=mask)


@triton.jit
def transpose_tiled_kernel(
    x_ptr, y_ptr,
    M, N,
    stride_xm, stride_xn,
    stride_ym, stride_yn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)

    x_block = tl.make_block_ptr(
        x_ptr,
        shape=(M, N),
        strides=(stride_xm, stride_xn),
        offsets=(pid_m * BLOCK_M, pid_n * BLOCK_N),
        block_shape=(BLOCK_M, BLOCK_N),
        order=(1, 0),
    )
    tile = tl.load(x_block, boundary_check=(0, 1))

    y_block = tl.make_block_ptr(
        y_ptr,
        shape=(N, M),
        strides=(stride_ym, stride_yn),
        offsets=(pid_n * BLOCK_N, pid_m * BLOCK_M),
        block_shape=(BLOCK_N, BLOCK_M),
        order=(1, 0),
    )
    tl.store(y_block, tl.trans(tile), boundary_check=(0, 1))


def transpose_naive(x, block_m=32, block_n=32):
    if x.ndim != 2 or not x.is_cuda:
        raise ValueError("x must be a CUDA matrix")
    M, N = x.shape
    y = torch.empty((N, M), device=x.device, dtype=x.dtype)
    grid = (triton.cdiv(M, block_m), triton.cdiv(N, block_n))
    transpose_naive_kernel[grid](
        x,
        y,
        M,
        N,
        x.stride(0),
        x.stride(1),
        y.stride(0),
        y.stride(1),
        BLOCK_M=block_m,
        BLOCK_N=block_n,
    )
    return y


def transpose_tiled(x, block_m=32, block_n=32):
    if x.ndim != 2 or not x.is_cuda:
        raise ValueError("x must be a CUDA matrix")
    M, N = x.shape
    y = torch.empty((N, M), device=x.device, dtype=x.dtype)
    grid = (triton.cdiv(M, block_m), triton.cdiv(N, block_n))
    transpose_tiled_kernel[grid](
        x,
        y,
        M,
        N,
        x.stride(0),
        x.stride(1),
        y.stride(0),
        y.stride(1),
        BLOCK_M=block_m,
        BLOCK_N=block_n,
    )
    return y

In [ ]:
transpose_input = torch.randn(
    (1025, 1537), device=device, dtype=torch.float32
)
naive_transpose = transpose_naive(transpose_input)
tiled_transpose = transpose_tiled(transpose_input)
transpose_reference = transpose_input.t().contiguous()

torch.testing.assert_close(
    naive_transpose, transpose_reference, rtol=0, atol=0
)
torch.testing.assert_close(
    tiled_transpose, transpose_reference, rtol=0, atol=0
)
print("Both transpose kernels match PyTorch.")

### Timing the two transpose kernels

The naive kernel and the block-pointer kernel move exactly the same bytes: one
read and one write per element. Their runtimes differ only in how many
global-memory transactions those addresses cost. Effective bandwidth divides the
useful bytes by the measured time, so the gap between the two rows is the price
of the scattered store.

In [ ]:
TRANSPOSE_BENCHMARK_CASES = [
    ("square 4096", 4096, 4096),
    ("tall", 8192, 1024),
    ("wide", 1024, 8192),
    ("boundary", 4097, 4095),
]


def launch_transpose(kernel, x, y, block_m=32, block_n=32):
    M, N = x.shape
    grid = (triton.cdiv(M, block_m), triton.cdiv(N, block_n))
    kernel[grid](
        x,
        y,
        M,
        N,
        x.stride(0),
        x.stride(1),
        y.stride(0),
        y.stride(1),
        BLOCK_M=block_m,
        BLOCK_N=block_n,
    )


def transpose_effective_gbps(M, N, itemsize, time_ms):
    moved_bytes = 2 * M * N * itemsize
    return moved_bytes / (time_ms * 1e-3) / 1e9


def transpose_providers(x, y):
    return [
        ("naive Triton", lambda: launch_transpose(transpose_naive_kernel, x, y)),
        ("tiled Triton", lambda: launch_transpose(transpose_tiled_kernel, x, y)),
        ("PyTorch copy of .t()", lambda: y.copy_(x.t())),
    ]


def run_transpose_benchmark(warmup_ms=100, measurement_ms=500):
    rows = []
    for name, M, N in TRANSPOSE_BENCHMARK_CASES:
        torch.manual_seed(0)
        x = torch.randn((M, N), device=device, dtype=torch.float32)
        y = torch.empty((N, M), device=device, dtype=torch.float32)
        itemsize = x.element_size()

        for provider, function in transpose_providers(x, y):
            function()
            torch.cuda.synchronize()
            p20_ms, p50_ms, p80_ms = triton.testing.do_bench(
                function,
                warmup=warmup_ms,
                rep=measurement_ms,
                quantiles=[0.2, 0.5, 0.8],
            )
            rows.append(
                {
                    "case": name,
                    "M": M,
                    "N": N,
                    "provider": provider,
                    "p20_ms": p20_ms,
                    "p50_ms": p50_ms,
                    "p80_ms": p80_ms,
                    "effective_gbps": transpose_effective_gbps(
                        M, N, itemsize, p50_ms
                    ),
                }
            )
    return rows


def print_transpose_benchmark(rows):
    header = (
        f"{'case':<12} {'provider':<22} {'p50 (ms)':>10} "
        f"{'GB/s':>9} {'vs naive':>9}"
    )
    print(header)
    print("-" * len(header))
    baseline = {}
    for row in rows:
        if row["provider"] == "naive Triton":
            baseline[row["case"]] = row["p50_ms"]
    for row in rows:
        speedup = baseline[row["case"]] / row["p50_ms"]
        print(
            f"{row['case']:<12} {row['provider']:<22} "
            f"{row['p50_ms']:>10.6f} {row['effective_gbps']:>9.1f} "
            f"{speedup:>8.2f}x"
        )


RUN_TRANSPOSE_BENCHMARK = False

if RUN_TRANSPOSE_BENCHMARK:
    transpose_benchmark_rows = run_transpose_benchmark()
    print_transpose_benchmark(transpose_benchmark_rows)
else:
    print("Set RUN_TRANSPOSE_BENCHMARK to True to time the transpose kernels.")

## Matrix multiplication in tiles

A tiled matrix multiplication program owns a rectangular tile of the output and accumulates one contribution for each slice of the contraction dimension. The small CPU example computes the top-left \(2 \times 2\) tile of a \(4 \times 4\) product. Its two loop iterations reproduce the numerical accumulation used to introduce `tl.dot` in the chapter.

In [ ]:
a_small = np.array(
    [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 16],
    ],
    dtype=np.int64,
)
b_small = np.array(
    [
        [1, 0, 2, 1],
        [0, 1, 3, 2],
        [1, 1, 0, 1],
        [2, 0, 1, 0],
    ],
    dtype=np.int64,
)

tile_size = 2
tile_accumulator = np.zeros((tile_size, tile_size), dtype=np.int64)

for k_start in range(0, a_small.shape[1], tile_size):
    a_tile = a_small[:tile_size, k_start : k_start + tile_size]
    b_tile = b_small[k_start : k_start + tile_size, :tile_size]
    partial = a_tile @ b_tile
    tile_accumulator += partial
    print(f"K slice {k_start}:{k_start + tile_size}")
    print("partial contribution:")
    print(partial)
    print("acc:")
    print(tile_accumulator)

full_product = a_small @ b_small
np.testing.assert_array_equal(
    tile_accumulator, full_product[:tile_size, :tile_size]
)
print("The accumulated tile matches the top-left tile of A @ B.")

## Scalar-output matrix multiplication

The scalar kernel launches one Triton program for every output element. Each program loads one row fragment from \(A\), one column fragment from \(B\), multiplies them elementwise, and reduces the fragment with `tl.sum`. Masked zero fills handle a final partial K block.

In [ ]:
@triton.jit
def naive_matmul_kernel(
    a_ptr,
    b_ptr,
    c_ptr,
    M,
    N,
    K,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)
    acc = 0.0

    for k_start in range(0, K, BLOCK_K):
        offs_k = k_start + tl.arange(0, BLOCK_K)
        a_ptrs = a_ptr + pid_m * stride_am + offs_k * stride_ak
        b_ptrs = b_ptr + offs_k * stride_bk + pid_n * stride_bn
        mask_k = offs_k < K
        a = tl.load(a_ptrs, mask=mask_k, other=0.0)
        b = tl.load(b_ptrs, mask=mask_k, other=0.0)
        a_fp32 = a.to(tl.float32)
        b_fp32 = b.to(tl.float32)
        acc += tl.sum(a_fp32 * b_fp32, axis=0)

    c_ptr_ij = c_ptr + pid_m * stride_cm + pid_n * stride_cn
    tl.store(c_ptr_ij, acc)


def scalar_matmul(a, b, block_k=256):
    if a.ndim != 2 or b.ndim != 2 or a.shape[1] != b.shape[0]:
        raise ValueError("a and b must be compatible matrices")
    if a.device != b.device or a.dtype != b.dtype or not a.is_cuda:
        raise ValueError("a and b must be matching CUDA tensors")

    M, K = a.shape
    _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    naive_matmul_kernel[(M, N)](
        a,
        b,
        c,
        M,
        N,
        K,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        c.stride(0),
        c.stride(1),
        BLOCK_K=block_k,
    )
    return c

In [ ]:
scalar_a = torch.randn((64, 96), device=device, dtype=torch.float16)
scalar_b = torch.randn((96, 80), device=device, dtype=torch.float16)
scalar_result = scalar_matmul(scalar_a, scalar_b, block_k=128)
scalar_reference = high_precision_reference(scalar_a, scalar_b)

torch.testing.assert_close(
    scalar_result, scalar_reference, rtol=1e-2, atol=1e-2
)
print("The scalar-output kernel matches the high-precision reference.")

### Benchmarking the scalar-output kernel

This cell is the complete benchmark listing printed in Chapter 5. It checks both providers before timing them and reports the measured latency and throughput for the active GPU.

In [ ]:
def launch_naive_matmul(a, b, c, block_k=256):
    M, K = a.shape
    _, N = b.shape
    naive_matmul_kernel[(M, N)](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_K=block_k,
    )


def launch_torch_matmul(a, b, c):
    torch.matmul(a, b, out=c)


def achieved_tflops(M, N, K, time_ms):
    return 2 * M * N * K / (time_ms * 1e9)


def benchmark_naive_matmul(M=1024, N=1024, K=1024):
    device = torch.device("cuda")
    dtype = torch.float16
    torch.manual_seed(0)

    a = torch.randn((M, K), device=device, dtype=dtype)
    b = torch.randn((K, N), device=device, dtype=dtype)
    naive_output = torch.empty((M, N), device=device, dtype=dtype)
    torch_output = torch.empty_like(naive_output)

    launch_naive_matmul(a, b, naive_output)
    launch_torch_matmul(a, b, torch_output)
    expected = torch.matmul(a.float(), b.float()).to(dtype)
    torch.cuda.synchronize()
    torch.testing.assert_close(
        naive_output, expected, rtol=1e-2, atol=1e-2
    )
    torch.testing.assert_close(
        torch_output, expected, rtol=1e-2, atol=1e-2
    )

    providers = [
        ("naive Triton", lambda: launch_naive_matmul(
            a, b, naive_output
        )),
        ("torch.matmul", lambda: launch_torch_matmul(
            a, b, torch_output
        )),
    ]
    results = {}
    for name, provider in providers:
        provider()
        torch.cuda.synchronize()
        p20_ms, p50_ms, p80_ms = triton.testing.do_bench(
            provider,
            warmup=100,
            rep=500,
            quantiles=[0.2, 0.5, 0.8],
        )
        results[name] = {
            "p20_ms": p20_ms,
            "p50_ms": p50_ms,
            "p80_ms": p80_ms,
            "tflops": achieved_tflops(M, N, K, p50_ms),
        }

    reference_tflops = results["torch.matmul"]["tflops"]
    print(f"GPU: {torch.cuda.get_device_name(device)}")
    print(f"PyTorch: {torch.__version__}")
    print(f"Triton: {triton.__version__}")
    print(f"CUDA runtime: {torch.version.cuda}")
    print(f"Shape: ({M}, {N}, {K}), dtype: {dtype}")
    for name, result in results.items():
        ratio = result["tflops"] / reference_tflops
        print(
            f"{name:14} "
            f"p20={result['p20_ms']:.4f} ms  "
            f"p50={result['p50_ms']:.4f} ms  "
            f"p80={result['p80_ms']:.4f} ms  "
            f"{result['tflops']:.4f} TFLOP/s  "
            f"ratio={ratio:.4f}"
        )


benchmark_naive_matmul()

## Tiled matrix multiplication with `tl.dot`

One program now computes a `BLOCK_M × BLOCK_N` output tile. During each K iteration, `a` has shape `[BLOCK_M, BLOCK_K]`, `b` has shape `[BLOCK_K, BLOCK_N]`, and the FP32 accumulator has shape `[BLOCK_M, BLOCK_N]`. The manual-pointer kernel advances element-pointer blocks, while the block-pointer kernel advances logical descriptors with `tl.advance`.

### Pointer construction used in the chapter

The first excerpt shows the manual pointer arrays. The second shows the block-pointer descriptors that replace them. These excerpts depend on kernel-local values, so they are shown here without executing them.

```python
a_ptrs = a_ptr + (offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak)
b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn)
```

```python
a_block_ptr = tl.make_block_ptr(
    base=a_ptr,
    shape=(M, K),
    strides=(stride_am, stride_ak),
    offsets=(pid_m * BLOCK_M, 0),
    block_shape=(BLOCK_M, BLOCK_K),
    order=(1, 0),
)
b_block_ptr = tl.make_block_ptr(
    base=b_ptr,
    shape=(K, N),
    strides=(stride_bk, stride_bn),
    offsets=(0, pid_n * BLOCK_N),
    block_shape=(BLOCK_K, BLOCK_N),
    order=(1, 0),
)
```

In [ ]:
import triton
import triton.language as tl

@triton.jit
def tiled_matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)

    offs_m = (
        pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    )
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    a_ptrs = (
        a_ptr
        + offs_m[:, None] * stride_am
        + offs_k[None, :] * stride_ak
    )
    b_ptrs = (
        b_ptr
        + offs_k[:, None] * stride_bk
        + offs_n[None, :] * stride_bn
    )

    acc = tl.zeros(
        (BLOCK_M, BLOCK_N), dtype=tl.float32
    )

    for k_start in range(0, K, BLOCK_K):
        a = tl.load(a_ptrs)
        b = tl.load(b_ptrs)
        acc = tl.dot(a, b, acc)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    c_ptrs = (
        c_ptr
        + offs_m[:, None] * stride_cm
        + offs_n[None, :] * stride_cn
    )
    tl.store(c_ptrs, acc)


@triton.jit
def block_ptr_tiled_matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)

    a_block_ptr = tl.make_block_ptr(
        base=a_ptr,
        shape=(M, K),
        strides=(stride_am, stride_ak),
        offsets=(pid_m * BLOCK_M, 0),
        block_shape=(BLOCK_M, BLOCK_K),
        order=(1, 0),
    )
    b_block_ptr = tl.make_block_ptr(
        base=b_ptr,
        shape=(K, N),
        strides=(stride_bk, stride_bn),
        offsets=(0, pid_n * BLOCK_N),
        block_shape=(BLOCK_K, BLOCK_N),
        order=(1, 0),
    )
    c_block = tl.make_block_ptr(
        base=c_ptr,
        shape=(M, N),
        strides=(stride_cm, stride_cn),
        offsets=(pid_m * BLOCK_M, pid_n * BLOCK_N),
        block_shape=(BLOCK_M, BLOCK_N),
        order=(1, 0),
    )

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for _ in range(0, K, BLOCK_K):
        a = tl.load(a_block_ptr)
        b = tl.load(b_block_ptr)
        acc = tl.dot(a, b, acc)
        a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))
        b_block_ptr = tl.advance(
            b_block_ptr, (BLOCK_K, 0)
        )

    output = acc.to(c_ptr.dtype.element_ty)
    tl.store(c_block, output)

In [ ]:
def check_aligned_matmul_inputs(a, b, block_m, block_n, block_k):
    if a.ndim != 2 or b.ndim != 2:
        raise ValueError("a and b must both be matrices")
    if a.shape[1] != b.shape[0]:
        raise ValueError("the inner matrix dimensions must match")
    if a.device != b.device or a.dtype != b.dtype:
        raise ValueError("a and b must have the same device and dtype")
    if not a.is_cuda or not a.is_contiguous() or not b.is_contiguous():
        raise ValueError("a and b must be contiguous GPU tensors")
    if a.dtype not in (torch.float16, torch.bfloat16):
        raise TypeError("this aligned example accepts FP16 or BF16 inputs")

    M, K = a.shape
    _, N = b.shape
    if M % block_m or N % block_n or K % block_k:
        raise ValueError(
            "M, N, and K must be divisible by their block sizes"
        )
    return M, N, K


def tiled_matmul(a, b, block_m=64, block_n=64, block_k=32):
    M, N, K = check_aligned_matmul_inputs(
        a, b, block_m, block_n, block_k
    )
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    grid = (
        triton.cdiv(M, block_m),
        triton.cdiv(N, block_n),
    )
    tiled_matmul_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=block_m,
        BLOCK_N=block_n,
        BLOCK_K=block_k,
    )
    return c


def block_ptr_tiled_matmul(a, b, block_m=64, block_n=64, block_k=32):
    M, N, K = check_aligned_matmul_inputs(
        a, b, block_m, block_n, block_k
    )
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    grid = (triton.cdiv(M, block_m), triton.cdiv(N, block_n))
    block_ptr_tiled_matmul_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=block_m,
        BLOCK_N=block_n,
        BLOCK_K=block_k,
    )
    return c

In [ ]:
torch.manual_seed(0)
matmul_a = torch.randn((512, 512), device=device, dtype=torch.float16)
matmul_b = torch.randn((512, 512), device=device, dtype=torch.float16)

manual_result = tiled_matmul(matmul_a, matmul_b)
block_pointer_result = block_ptr_tiled_matmul(matmul_a, matmul_b)
matmul_reference = high_precision_reference(matmul_a, matmul_b)

torch.testing.assert_close(
    manual_result, matmul_reference, rtol=1e-2, atol=1e-2
)
torch.testing.assert_close(
    block_pointer_result, matmul_reference, rtol=1e-2, atol=1e-2
)
torch.testing.assert_close(
    block_pointer_result, manual_result, rtol=1e-2, atol=1e-2
)
print("Both tiled kernels match the high-precision reference.")

## Boundary-safe autotuned matrix multiplication

The final kernel assigns one output tile to each Triton program, groups program IDs along M to encourage L2 reuse, masks all three boundary conditions, accumulates with `tl.dot`, and lets Triton select a configuration for each `(M, N, K)` key.

In [ ]:
def matmul_autotune_configs():
    return [
        triton.Config(
            {"BLOCK_M": 128, "BLOCK_N": 256, "BLOCK_K": 64, "GROUP_M": 8},
            num_warps=8, num_stages=3,
        ),
        triton.Config(
            {"BLOCK_M": 64, "BLOCK_N": 256, "BLOCK_K": 32, "GROUP_M": 8},
            num_warps=4, num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 128, "BLOCK_N": 128, "BLOCK_K": 32, "GROUP_M": 8},
            num_warps=4, num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 128, "BLOCK_N": 64, "BLOCK_K": 32, "GROUP_M": 8},
            num_warps=4, num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 64, "BLOCK_N": 128, "BLOCK_K": 32, "GROUP_M": 8},
            num_warps=4, num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 64, "BLOCK_N": 64, "BLOCK_K": 32, "GROUP_M": 8},
            num_warps=4, num_stages=4,
        ),
    ]


@triton.autotune(
    configs=matmul_autotune_configs(),
    key=["M", "N", "K"],
)
@triton.jit
def optimized_matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    GROUP_M: tl.constexpr,
):
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    num_pid_in_group = GROUP_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_M)
    pid_in_group = pid % num_pid_in_group
    pid_m = first_pid_m + pid_in_group % group_size_m
    pid_n = pid_in_group // group_size_m

    tl.assume(pid_m >= 0)
    tl.assume(pid_n >= 0)
    tl.assume(stride_am > 0)
    tl.assume(stride_ak > 0)
    tl.assume(stride_bk > 0)
    tl.assume(stride_bn > 0)
    tl.assume(stride_cm > 0)
    tl.assume(stride_cn > 0)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    offs_m = tl.max_contiguous(
        tl.multiple_of(offs_m, BLOCK_M), BLOCK_M
    )
    offs_n = tl.max_contiguous(
        tl.multiple_of(offs_n, BLOCK_N), BLOCK_N
    )

    a_ptrs = (
        a_ptr
        + offs_m[:, None] * stride_am
        + offs_k[None, :] * stride_ak
    )
    b_ptrs = (
        b_ptr
        + offs_k[:, None] * stride_bk
        + offs_n[None, :] * stride_bn
    )

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k_tile in range(0, tl.cdiv(K, BLOCK_K)):
        valid_k = offs_k < K - k_tile * BLOCK_K
        a_mask = (offs_m[:, None] < M) & valid_k[None, :]
        b_mask = valid_k[:, None] & (offs_n[None, :] < N)
        a = tl.load(a_ptrs, mask=a_mask, other=0.0)
        b = tl.load(b_ptrs, mask=b_mask, other=0.0)
        acc = tl.dot(a, b, acc)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    c_ptrs = (
        c_ptr
        + offs_m[:, None] * stride_cm
        + offs_n[None, :] * stride_cn
    )
    c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(c_ptrs, acc, mask=c_mask)


def optimized_matmul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    if a.ndim != 2 or b.ndim != 2:
        raise ValueError("a and b must both be matrices")
    if a.shape[1] != b.shape[0]:
        raise ValueError("the inner matrix dimensions must match")
    if a.device != b.device or a.dtype != b.dtype:
        raise ValueError("a and b must have the same device and dtype")
    if not a.is_cuda or not a.is_contiguous() or not b.is_contiguous():
        raise ValueError("a and b must be contiguous GPU tensors")
    if a.dtype not in (torch.float16, torch.bfloat16):
        raise TypeError("this kernel accepts FP16 or BF16 inputs")

    M, K = a.shape
    _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    grid = lambda meta: (
        triton.cdiv(M, meta["BLOCK_M"])
        * triton.cdiv(N, meta["BLOCK_N"]),
    )
    optimized_matmul_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
    )
    return c

In [ ]:
def launch_scalar_matmul(a, b, c):
    M, K = a.shape
    _, N = b.shape
    naive_matmul_kernel[(M, N)](
        a,
        b,
        c,
        M,
        N,
        K,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        c.stride(0),
        c.stride(1),
        BLOCK_K=256,
    )

def launch_fixed_tiled_matmul(a, b, c):
    M, K = a.shape
    _, N = b.shape
    tiled_matmul_kernel[(triton.cdiv(M, 64), triton.cdiv(N, 64))](
        a,
        b,
        c,
        M,
        N,
        K,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        c.stride(0),
        c.stride(1),
        BLOCK_M=64,
        BLOCK_N=64,
        BLOCK_K=32,
    )

def launch_fixed_block_ptr_matmul(a, b, c):
    M, K = a.shape
    _, N = b.shape
    block_ptr_tiled_matmul_kernel[(triton.cdiv(M, 64), triton.cdiv(N, 64))](
        a,
        b,
        c,
        M,
        N,
        K,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        c.stride(0),
        c.stride(1),
        BLOCK_M=64,
        BLOCK_N=64,
        BLOCK_K=32,
    )

def launch_optimized_matmul(a, b, c):
    M, K = a.shape
    _, N = b.shape
    grid = lambda meta: (
        triton.cdiv(M, meta["BLOCK_M"]) * triton.cdiv(N, meta["BLOCK_N"]),
    )
    optimized_matmul_kernel[grid](
        a,
        b,
        c,
        M,
        N,
        K,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        c.stride(0),
        c.stride(1),
    )

def launch_reference_matmul(a, b, c):
    torch.matmul(a, b, out=c)


def synchronize_matmul(device):
    torch.cuda.synchronize(device)

## Correctness across shapes and dtypes

The suite compares every Triton result with a high-precision PyTorch reference. It covers aligned, rectangular, remainder, sub-tile, FP16, and BF16 cases. The scalar and final kernels run for every supported case. The two fixed tiled kernels run only when their alignment contract holds.

In [ ]:
MATMUL_CORRECTNESS_CASES = [
    ("aligned square", 256, 256, 256),
    ("M remainder", 257, 256, 256),
    ("N remainder", 256, 259, 256),
    ("K remainder", 256, 256, 263),
    ("all remainders", 257, 259, 263),
    ("small boundary", 31, 47, 29),
    ("tall", 769, 193, 257),
    ("wide", 193, 769, 257),
]

MATMUL_CORRECTNESS_DTYPES = [
    (torch.float16, 1e-2, 1e-2),
    (torch.bfloat16, 3e-2, 3e-2),
]


def high_precision_reference(a, b):
    previous_precision = torch.get_float32_matmul_precision()
    try:
        torch.set_float32_matmul_precision("highest")
        reference = torch.matmul(a.float(), b.float())
    finally:
        torch.set_float32_matmul_precision(previous_precision)
    return reference.to(dtype=a.dtype)


def run_matmul_correctness_suite():
    device = triton.runtime.driver.active.get_active_torch_device()
    records = []
    for dtype, rtol, atol in MATMUL_CORRECTNESS_DTYPES:
        for seed, (name, M, N, K) in enumerate(MATMUL_CORRECTNESS_CASES):
            torch.manual_seed(seed)
            a = torch.randn((M, K), device=device, dtype=dtype)
            b = torch.randn((K, N), device=device, dtype=dtype)
            expected = high_precision_reference(a, b)
            actual = optimized_matmul(a, b)
            torch.testing.assert_close(
                actual, expected, rtol=rtol, atol=atol
            )

            if M % 64 == 0 and N % 64 == 0 and K % 32 == 0:
                manual = tiled_matmul(a, b)
                block_ptr = block_ptr_tiled_matmul(a, b)
                torch.testing.assert_close(
                    manual, expected, rtol=rtol, atol=atol
                )
                torch.testing.assert_close(
                    block_ptr, expected, rtol=rtol, atol=atol
                )

            records.append((name, str(dtype), M, N, K, "pass"))
    return records


for record in run_matmul_correctness_suite():
    print(record)

## H200 matrix-multiplication benchmark

The benchmark preallocates each output, launches once before timing, and measures the 20th, 50th, and 80th percentile with a 100 ms warmup and a 500 ms measurement interval. Set `RUN_LIVE_H200_BENCHMARK` to `True` to execute three notebook passes. The manuscript numbers below came from three fresh Python processes, so a same-kernel notebook run is a reproduction rather than the original process-isolated measurement.

In [ ]:
MATMUL_BENCHMARK_CASES = [
    ("square 1024", 1024, 1024, 1024, "fp16"),
    ("square 4096", 4096, 4096, 4096, "fp16"),
    ("tall", 4096, 1024, 4096, "fp16"),
    ("wide", 1024, 4096, 4096, "fp16"),
    ("K-heavy BF16", 1024, 1024, 8192, "bf16"),
    ("boundary", 1001, 1537, 509, "fp16"),
]


def benchmark_dtype(name):
    return {"fp16": torch.float16, "bf16": torch.bfloat16}[name]


def make_matmul_inputs(M, N, K, dtype, device, seed):
    torch.manual_seed(seed)
    a = torch.randn((M, K), device=device, dtype=dtype)
    b = torch.randn((K, N), device=device, dtype=dtype)
    return a, b


def make_matmul_provider(name, launch, a, b):
    output = torch.empty(
        (a.shape[0], b.shape[1]),
        device=a.device,
        dtype=a.dtype,
    )

    def run():
        launch(a, b, output)

    return name, run


def matmul_providers(a, b, include_slow_scalar):
    M, K = a.shape
    _, N = b.shape
    providers = []

    if include_slow_scalar:
        providers.append(
            make_matmul_provider(
                "scalar tl.sum",
                launch_scalar_matmul,
                a,
                b,
            )
        )

    if M % 64 == 0 and N % 64 == 0 and K % 32 == 0:
        providers.extend(
            [
                make_matmul_provider(
                    "fixed manual tl.dot",
                    launch_fixed_tiled_matmul,
                    a,
                    b,
                ),
                make_matmul_provider(
                    "fixed block pointer",
                    launch_fixed_block_ptr_matmul,
                    a,
                    b,
                ),
            ]
        )

    providers.extend(
        [
            make_matmul_provider(
                "optimized Triton",
                launch_optimized_matmul,
                a,
                b,
            ),
            make_matmul_provider(
                "PyTorch reference",
                launch_reference_matmul,
                a,
                b,
            ),
        ]
    )
    return providers


def selected_matmul_config():
    config = getattr(optimized_matmul_kernel, "best_config", None)
    if config is None:
        return ""

    values = config.kwargs
    return (
        f"BM={values['BLOCK_M']},BN={values['BLOCK_N']},"
        f"BK={values['BLOCK_K']},GM={values['GROUP_M']},"
        f"warps={config.num_warps},stages={config.num_stages}"
    )


def collect_matmul_environment():
    device = triton.runtime.driver.active.get_active_torch_device()
    target = triton.runtime.driver.active.get_current_target()
    capability = torch.cuda.get_device_capability(device)
    return {
        "gpu": torch.cuda.get_device_name(device),
        "compute_capability": f"{capability[0]}.{capability[1]}",
        "total_memory_gib": (
            torch.cuda.get_device_properties(device).total_memory / 2**30
        ),
        "backend": target.backend,
        "architecture": str(target.arch),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "triton": triton.__version__,
        "backend_runtime": torch.version.hip or torch.version.cuda,
    }


def run_matmul_benchmark_pass(
    pass_index=1,
    warmup_ms=100,
    measurement_ms=500,
    include_slow_scalar=True,
):
    device = triton.runtime.driver.active.get_active_torch_device()
    environment = collect_matmul_environment()
    rows = []

    for seed, (case_name, M, N, K, dtype_name) in enumerate(
        MATMUL_BENCHMARK_CASES
    ):
        dtype = benchmark_dtype(dtype_name)
        if dtype == torch.bfloat16 and not torch.cuda.is_bf16_supported():
            continue

        a, b = make_matmul_inputs(M, N, K, dtype, device, seed)
        case_rows = []

        for provider, function in matmul_providers(
            a,
            b,
            include_slow_scalar,
        ):
            print(f"Pass {pass_index}, {case_name}: {provider}")
            function()
            synchronize_matmul(device)
            p20_ms, p50_ms, p80_ms = triton.testing.do_bench(
                function,
                warmup=warmup_ms,
                rep=measurement_ms,
                quantiles=[0.2, 0.5, 0.8],
            )
            row = {
                "pass": pass_index,
                "case": case_name,
                "M": M,
                "N": N,
                "K": K,
                "dtype": dtype_name,
                "provider": provider,
                "p20_ms": p20_ms,
                "p50_ms": p50_ms,
                "p80_ms": p80_ms,
                "tflops": 2 * M * N * K / p50_ms * 1e-9,
                "relative_to_reference": 0.0,
                "autotune_config": "",
                "gpu": environment["gpu"],
                "backend": environment["backend"],
                "architecture": environment["architecture"],
                "torch": environment["torch"],
                "triton": environment["triton"],
            }
            if provider == "optimized Triton":
                row["autotune_config"] = selected_matmul_config()
            case_rows.append(row)

        reference_tflops = next(
            row["tflops"]
            for row in case_rows
            if row["provider"] == "PyTorch reference"
        )
        for row in case_rows:
            row["relative_to_reference"] = (
                row["tflops"] / reference_tflops
            )
        rows.extend(case_rows)

    return rows


def summarize_matmul_benchmark(rows):
    groups = {}
    for row in rows:
        key = (
            row["case"],
            row["M"],
            row["N"],
            row["K"],
            row["dtype"],
            row["provider"],
        )
        groups.setdefault(key, []).append(row)

    summaries = []
    for key, group in sorted(groups.items()):
        case_name, M, N, K, dtype_name, provider = key
        summary = {
            "case": case_name,
            "M": M,
            "N": N,
            "K": K,
            "dtype": dtype_name,
            "provider": provider,
            "notebook_passes": len(group),
        }
        for field in (
            "p20_ms",
            "p50_ms",
            "p80_ms",
            "tflops",
            "relative_to_reference",
        ):
            values = [row[field] for row in group]
            summary[f"{field}_median"] = statistics.median(values)
            summary[f"{field}_min"] = min(values)
            summary[f"{field}_max"] = max(values)

        summary["autotune_configs"] = "; ".join(
            sorted(
                {
                    row["autotune_config"]
                    for row in group
                    if row["autotune_config"]
                }
            )
        )
        summaries.append(summary)

    return summaries


def run_matmul_profile_target(
    provider="optimized",
    M=4096,
    N=4096,
    K=4096,
    dtype_name="fp16",
    repetitions=10,
):
    device = triton.runtime.driver.active.get_active_torch_device()
    dtype = benchmark_dtype(dtype_name)
    a, b = make_matmul_inputs(M, N, K, dtype, device, 0)
    c = torch.empty((M, N), device=device, dtype=dtype)
    launch = {
        "scalar": launch_scalar_matmul,
        "optimized": launch_optimized_matmul,
    }[provider]

    launch(a, b, c)
    synchronize_matmul(device)
    for _ in range(repetitions):
        launch(a, b, c)
    synchronize_matmul(device)
    return c

In [ ]:
RUN_LIVE_H200_BENCHMARK = False

if RUN_LIVE_H200_BENCHMARK:
    live_matmul_rows = []
    for pass_index in range(1, 4):
        live_matmul_rows.extend(
            run_matmul_benchmark_pass(
                pass_index=pass_index,
                warmup_ms=100,
                measurement_ms=500,
                include_slow_scalar=True,
            )
        )
    live_matmul_summary = summarize_matmul_benchmark(live_matmul_rows)
    print(f"Collected {len(live_matmul_summary)} summarized rows.")
else:
    print(
        "Set RUN_LIVE_H200_BENCHMARK to True to run the complete benchmark."
    )

## Recorded NVIDIA H200 results

These are the complete 28-row benchmark summary and 36-row correctness result used by the chapter. Keeping them in the notebook preserves the measurements without a CSV, JSON file, ZIP, or result archive. The timings remain specific to the recorded H200 and software stack.

In [ ]:
RECORDED_H200_ENVIRONMENT = {
    "gpu": "NVIDIA H200",
    "compute_capability": "9.0",
    "memory_mib": 143771,
    "power_limit_w": 700,
    "driver": "595.58.03",
    "python": "3.10.20",
    "torch": "2.11.0+cu130",
    "triton": "3.6.0",
    "torch_cuda_runtime": "13.0",
}

RECORDED_H200_BENCHMARK_RESULTS = [
    {
        "case": "K-heavy BF16",
        "M": 1024,
        "N": 1024,
        "K": 8192,
        "dtype": "bf16",
        "provider": "PyTorch reference",
        "independent_runs": 3,
        "p20_ms_median": 0.0307839997112751,
        "p20_ms_min": 0.03046399913728237,
        "p20_ms_max": 0.030848000198602676,
        "p50_ms_median": 0.03110400028526783,
        "p50_ms_min": 0.03081599995493889,
        "p50_ms_max": 0.031136000528931618,
        "p80_ms_median": 0.03209599852561951,
        "p80_ms_min": 0.03177599981427193,
        "p80_ms_max": 0.03209599852561951,
        "tflops_median": 552.336324152399,
        "tflops_min": 551.7686566081742,
        "tflops_max": 557.49835180171,
        "relative_to_reference_median": 1,
        "relative_to_reference_min": 1,
        "relative_to_reference_max": 1,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "K-heavy BF16",
        "M": 1024,
        "N": 1024,
        "K": 8192,
        "dtype": "bf16",
        "provider": "fixed block pointer",
        "independent_runs": 3,
        "p20_ms_median": 0.09647999703884125,
        "p20_ms_min": 0.09628800302743912,
        "p20_ms_max": 0.09651199728250504,
        "p50_ms_median": 0.09689600020647049,
        "p50_ms_min": 0.09676799923181534,
        "p50_ms_max": 0.09689600020647049,
        "p80_ms_median": 0.09731200337409973,
        "p80_ms_min": 0.09721600264310837,
        "p80_ms_max": 0.09734400361776352,
        "tflops_median": 177.30215021664813,
        "tflops_min": 177.30215021664813,
        "tflops_max": 177.53667865803732,
        "relative_to_reference_median": 0.3210039652719408,
        "relative_to_reference_min": 0.3180317029523894,
        "relative_to_reference_max": 0.32175926727949483,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "K-heavy BF16",
        "M": 1024,
        "N": 1024,
        "K": 8192,
        "dtype": "bf16",
        "provider": "fixed manual tl.dot",
        "independent_runs": 3,
        "p20_ms_median": 0.09663999825716019,
        "p20_ms_min": 0.09638399630784988,
        "p20_ms_max": 0.09663999825716019,
        "p50_ms_median": 0.09702400118112564,
        "p50_ms_min": 0.0968639999628067,
        "p50_ms_max": 0.09705600142478943,
        "p80_ms_median": 0.09743999689817429,
        "p80_ms_min": 0.09731200337409973,
        "p80_ms_max": 0.09743999689817429,
        "tflops_median": 177.06824058851586,
        "tflops_min": 177.0098595841393,
        "tflops_max": 177.36072421742475,
        "relative_to_reference_median": 0.3205804739716154,
        "relative_to_reference_min": 0.31750741327231374,
        "relative_to_reference_max": 0.32144037558728783,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "K-heavy BF16",
        "M": 1024,
        "N": 1024,
        "K": 8192,
        "dtype": "bf16",
        "provider": "optimized Triton",
        "independent_runs": 3,
        "p20_ms_median": 0.07168000191450119,
        "p20_ms_min": 0.0716480016708374,
        "p20_ms_max": 0.0719359964132309,
        "p50_ms_median": 0.07248000055551529,
        "p50_ms_min": 0.0724480003118515,
        "p50_ms_max": 0.07273600250482559,
        "p80_ms_median": 0.07334399968385696,
        "p80_ms_min": 0.07331199944019318,
        "p80_ms_max": 0.07356800138950348,
        "tflops_median": 237.0290984040661,
        "tflops_min": 236.1948497631585,
        "tflops_max": 237.13379403226412,
        "relative_to_reference_median": 0.42806862429463227,
        "relative_to_reference_min": 0.42535335443755257,
        "relative_to_reference_max": 0.42913907349440544,
        "autotune_configs_across_runs": "BM=64,BN=128,BK=32,GM=8,warps=4,stages=4"
    },
    {
        "case": "K-heavy BF16",
        "M": 1024,
        "N": 1024,
        "K": 8192,
        "dtype": "bf16",
        "provider": "scalar tl.sum",
        "independent_runs": 3,
        "p20_ms_median": 40.82738571166992,
        "p20_ms_min": 40.825556182861334,
        "p20_ms_max": 40.82981185913086,
        "p50_ms_median": 40.83180618286133,
        "p50_ms_min": 40.829872131347656,
        "p50_ms_max": 40.835391998291016,
        "p80_ms_median": 40.83836822509765,
        "p80_ms_min": 40.8359878540039,
        "p80_ms_max": 40.86206588745117,
        "tflops_median": 0.4207472259997906,
        "tflops_min": 0.4207102795711864,
        "tflops_max": 0.4207671561824446,
        "relative_to_reference_median": 0.0007616922175393724,
        "relative_to_reference_min": 0.0007547057756135595,
        "relative_to_reference_max": 0.0007625789380081491,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "boundary",
        "M": 1001,
        "N": 1537,
        "K": 509,
        "dtype": "fp16",
        "provider": "PyTorch reference",
        "independent_runs": 3,
        "p20_ms_median": 0.027807999402284622,
        "p20_ms_min": 0.027775999158620834,
        "p20_ms_max": 0.028192000463604927,
        "p50_ms_median": 0.02812799997627735,
        "p50_ms_min": 0.028063999488949776,
        "p50_ms_max": 0.028416000306606293,
        "p80_ms_median": 0.028543999418616295,
        "p80_ms_min": 0.028511999174952507,
        "p80_ms_max": 0.028831999748945236,
        "tflops_median": 55.682262063457436,
        "tflops_min": 55.1179141716111,
        "tflops_max": 55.809246526558155,
        "relative_to_reference_median": 1,
        "relative_to_reference_min": 1,
        "relative_to_reference_max": 1,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "boundary",
        "M": 1001,
        "N": 1537,
        "K": 509,
        "dtype": "fp16",
        "provider": "optimized Triton",
        "independent_runs": 3,
        "p20_ms_median": 0.02470399998128414,
        "p20_ms_min": 0.024639999493956566,
        "p20_ms_max": 0.024800000712275505,
        "p50_ms_median": 0.024960000067949295,
        "p50_ms_min": 0.02489599958062172,
        "p50_ms_max": 0.02505600079894066,
        "p80_ms_median": 0.025407999753952026,
        "p80_ms_min": 0.02534399926662445,
        "p80_ms_max": 0.02550400048494339,
        "tflops_median": 62.74962587084164,
        "tflops_min": 62.50920402533746,
        "tflops_max": 62.91093719406655,
        "relative_to_reference_median": 1.1272493557878243,
        "relative_to_reference_min": 1.1269230729047968,
        "relative_to_reference_max": 1.1340995929329507,
        "autotune_configs_across_runs": "BM=64,BN=128,BK=32,GM=8,warps=4,stages=4"
    },
    {
        "case": "boundary",
        "M": 1001,
        "N": 1537,
        "K": 509,
        "dtype": "fp16",
        "provider": "scalar tl.sum",
        "independent_runs": 3,
        "p20_ms_median": 1.62716805934906,
        "p20_ms_min": 1.6265920400619507,
        "p20_ms_max": 1.6280319690704346,
        "p50_ms_median": 1.6276799440383911,
        "p50_ms_min": 1.6269760131835938,
        "p50_ms_max": 1.6285760402679443,
        "p80_ms_median": 1.6281280517578125,
        "p80_ms_min": 1.6273599863052368,
        "p80_ms_max": 1.6301695823669435,
        "tflops_median": 0.9622473212479776,
        "tflops_min": 0.9617178610476876,
        "tflops_max": 0.9626636491925104,
        "relative_to_reference_median": 0.01728851547186473,
        "relative_to_reference_min": 0.01723223159069226,
        "relative_to_reference_max": 0.01745797778653225,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "square 1024",
        "M": 1024,
        "N": 1024,
        "K": 1024,
        "dtype": "fp16",
        "provider": "PyTorch reference",
        "independent_runs": 3,
        "p20_ms_median": 0.009983999654650688,
        "p20_ms_min": 0.009983999654650688,
        "p20_ms_max": 0.010080000385642052,
        "p50_ms_median": 0.010143999941647053,
        "p50_ms_min": 0.010111999697983265,
        "p50_ms_max": 0.010208000428974628,
        "p80_ms_median": 0.010304000228643417,
        "p80_ms_min": 0.010304000228643417,
        "p80_ms_max": 0.010400000028312206,
        "tflops_median": 211.69988765312624,
        "tflops_min": 210.37260557949546,
        "tflops_max": 212.36982912769406,
        "relative_to_reference_median": 1,
        "relative_to_reference_min": 1,
        "relative_to_reference_max": 1,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "square 1024",
        "M": 1024,
        "N": 1024,
        "K": 1024,
        "dtype": "fp16",
        "provider": "fixed block pointer",
        "independent_runs": 3,
        "p20_ms_median": 0.016672000288963318,
        "p20_ms_min": 0.01664000004529953,
        "p20_ms_max": 0.01679999940097332,
        "p50_ms_median": 0.01696000061929226,
        "p50_ms_min": 0.01692800037562847,
        "p50_ms_max": 0.017023999243974686,
        "p80_ms_median": 0.017343999817967415,
        "p80_ms_min": 0.017311999574303627,
        "p80_ms_max": 0.017376000061631203,
        "tflops_median": 126.62049348967622,
        "tflops_min": 126.14448680500618,
        "tflops_max": 126.85985351771191,
        "relative_to_reference_median": 0.5992438395884927,
        "relative_to_reference_min": 0.5939849710438757,
        "relative_to_reference_max": 0.6018867957683253,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "square 1024",
        "M": 1024,
        "N": 1024,
        "K": 1024,
        "dtype": "fp16",
        "provider": "fixed manual tl.dot",
        "independent_runs": 3,
        "p20_ms_median": 0.016896000131964684,
        "p20_ms_min": 0.016767999157309532,
        "p20_ms_max": 0.01696000061929226,
        "p50_ms_median": 0.01711999997496605,
        "p50_ms_min": 0.01708799973130226,
        "p50_ms_max": 0.017184000462293625,
        "p80_ms_median": 0.017503999173641205,
        "p80_ms_min": 0.01744000054895878,
        "p80_ms_max": 0.017503999173641205,
        "tflops_median": 125.43712915538475,
        "tflops_min": 124.96994822085601,
        "tflops_max": 125.67203193865818,
        "relative_to_reference_median": 0.5906541888300043,
        "relative_to_reference_min": 0.5903165542799972,
        "relative_to_reference_max": 0.5973783116507977,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "square 1024",
        "M": 1024,
        "N": 1024,
        "K": 1024,
        "dtype": "fp16",
        "provider": "optimized Triton",
        "independent_runs": 3,
        "p20_ms_median": 0.01398400031030178,
        "p20_ms_min": 0.01398400031030178,
        "p20_ms_max": 0.014015999622642994,
        "p50_ms_median": 0.014303999952971935,
        "p50_ms_min": 0.014271999709308147,
        "p50_ms_max": 0.014336000196635723,
        "p80_ms_median": 0.01462399959564209,
        "p80_ms_min": 0.014592000283300877,
        "p80_ms_max": 0.014720000326633453,
        "tflops_median": 150.13168729449126,
        "tflops_min": 149.7965693739288,
        "tflops_max": 150.46830799747136,
        "relative_to_reference_median": 0.7107623422267288,
        "relative_to_reference_min": 0.7069351042525907,
        "relative_to_reference_max": 0.7120535915847835,
        "autotune_configs_across_runs": "BM=64,BN=128,BK=32,GM=8,warps=4,stages=4"
    },
    {
        "case": "square 1024",
        "M": 1024,
        "N": 1024,
        "K": 1024,
        "dtype": "fp16",
        "provider": "scalar tl.sum",
        "independent_runs": 3,
        "p20_ms_median": 5.4712383270263665,
        "p20_ms_min": 5.470880031585693,
        "p20_ms_max": 5.471366596221924,
        "p50_ms_median": 5.471551895141602,
        "p50_ms_min": 5.471199989318848,
        "p50_ms_max": 5.4718239307403564,
        "p80_ms_median": 5.471839904785156,
        "p80_ms_min": 5.471424102783203,
        "p80_ms_max": 5.472064018249512,
        "tflops_median": 0.3924816375966081,
        "tflops_min": 0.39246212509426237,
        "tflops_max": 0.39250688188924293,
        "relative_to_reference_median": 0.0018540722257367086,
        "relative_to_reference_min": 0.001848012623574143,
        "relative_to_reference_max": 0.001865649933438208,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "square 4096",
        "M": 4096,
        "N": 4096,
        "K": 4096,
        "dtype": "fp16",
        "provider": "PyTorch reference",
        "independent_runs": 3,
        "p20_ms_median": 0.20025600492954254,
        "p20_ms_min": 0.19992319941520692,
        "p20_ms_max": 0.20134399831295013,
        "p50_ms_median": 0.2067679986357689,
        "p50_ms_min": 0.20528000593185425,
        "p50_ms_max": 0.20825600624084473,
        "p80_ms_median": 0.2138880044221878,
        "p80_ms_min": 0.21174399554729462,
        "p80_ms_max": 0.21542400121688843,
        "tflops_median": 664.7012805598845,
        "tflops_min": 659.9519310528508,
        "tflops_max": 669.5194344334973,
        "relative_to_reference_median": 1,
        "relative_to_reference_min": 1,
        "relative_to_reference_max": 1,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "square 4096",
        "M": 4096,
        "N": 4096,
        "K": 4096,
        "dtype": "fp16",
        "provider": "fixed block pointer",
        "independent_runs": 3,
        "p20_ms_median": 0.4251840114593506,
        "p20_ms_min": 0.42447999119758606,
        "p20_ms_max": 0.4253184080123901,
        "p50_ms_median": 0.43089599907398224,
        "p50_ms_min": 0.429967999458313,
        "p50_ms_max": 0.4320160001516342,
        "p80_ms_median": 0.4572480022907257,
        "p80_ms_min": 0.4548479914665222,
        "p80_ms_max": 0.46344320178031917,
        "tflops_median": 318.9608484816833,
        "tflops_min": 318.13394277934157,
        "tflops_max": 319.6492614453863,
        "relative_to_reference_median": 0.4798559259777857,
        "relative_to_reference_min": 0.47743089297452906,
        "relative_to_reference_max": 0.4820562344166616,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "square 4096",
        "M": 4096,
        "N": 4096,
        "K": 4096,
        "dtype": "fp16",
        "provider": "fixed manual tl.dot",
        "independent_runs": 3,
        "p20_ms_median": 0.4191359877586365,
        "p20_ms_min": 0.4190079867839813,
        "p20_ms_max": 0.41992959976196287,
        "p50_ms_median": 0.4222399890422821,
        "p50_ms_min": 0.42182400822639465,
        "p50_ms_max": 0.42297598719596863,
        "p80_ms_median": 0.42520960569381716,
        "p80_ms_min": 0.42467200756073,
        "p80_ms_max": 0.425651204586029,
        "tflops_median": 325.4996140553547,
        "tflops_min": 324.9332293852495,
        "tflops_max": 325.8206047822578,
        "relative_to_reference_median": 0.48969307503241627,
        "relative_to_reference_min": 0.4866484645930338,
        "relative_to_reference_max": 0.4923589341830836,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "square 4096",
        "M": 4096,
        "N": 4096,
        "K": 4096,
        "dtype": "fp16",
        "provider": "optimized Triton",
        "independent_runs": 3,
        "p20_ms_median": 0.22169600427150726,
        "p20_ms_min": 0.22115199267864227,
        "p20_ms_max": 0.2226880043745041,
        "p50_ms_median": 0.22387200593948364,
        "p50_ms_min": 0.22246399521827698,
        "p50_ms_max": 0.22511999309062958,
        "p80_ms_median": 0.22780799865722656,
        "p80_ms_min": 0.22697600722312927,
        "p80_ms_max": 0.23160320222377778,
        "tflops_median": 613.9175503218212,
        "tflops_min": 610.5142043810803,
        "tflops_max": 617.8031341078263,
        "relative_to_reference_median": 0.9227560879253193,
        "relative_to_reference_min": 0.9184790555342968,
        "relative_to_reference_max": 0.9302458579709147,
        "autotune_configs_across_runs": "BM=128,BN=256,BK=64,GM=8,warps=8,stages=3"
    },
    {
        "case": "square 4096",
        "M": 4096,
        "N": 4096,
        "K": 4096,
        "dtype": "fp16",
        "provider": "scalar tl.sum",
        "independent_runs": 3,
        "p20_ms_median": 331.1535339355469,
        "p20_ms_min": 330.9361572265625,
        "p20_ms_max": 331.16778564453125,
        "p50_ms_median": 331.1535339355469,
        "p50_ms_min": 330.9361572265625,
        "p50_ms_max": 331.16778564453125,
        "p80_ms_median": 331.1535339355469,
        "p80_ms_min": 330.9361572265625,
        "p80_ms_max": 331.16778564453125,
        "tflops_median": 0.4150309128174669,
        "tflops_min": 0.415013052083285,
        "tflops_max": 0.41530352749550964,
        "relative_to_reference_median": 0.0006243871118585513,
        "relative_to_reference_min": 0.0006203009294971576,
        "relative_to_reference_max": 0.0006288534551617966,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "tall",
        "M": 4096,
        "N": 1024,
        "K": 4096,
        "dtype": "fp16",
        "provider": "PyTorch reference",
        "independent_runs": 3,
        "p20_ms_median": 0.04896000027656555,
        "p20_ms_min": 0.04872959852218628,
        "p20_ms_max": 0.049056001007556915,
        "p50_ms_median": 0.04931199923157692,
        "p50_ms_min": 0.04915200173854828,
        "p50_ms_max": 0.04944000020623207,
        "p80_ms_median": 0.04982399940490723,
        "p80_ms_min": 0.04979199916124344,
        "p80_ms_max": 0.04995200037956238,
        "tflops_median": 696.7825053419808,
        "tflops_min": 694.9785239618353,
        "tflops_max": 699.0506419406477,
        "relative_to_reference_median": 1,
        "relative_to_reference_min": 1,
        "relative_to_reference_max": 1,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "tall",
        "M": 4096,
        "N": 1024,
        "K": 4096,
        "dtype": "fp16",
        "provider": "fixed block pointer",
        "independent_runs": 3,
        "p20_ms_median": 0.10745599865913391,
        "p20_ms_min": 0.10729599744081497,
        "p20_ms_max": 0.1074879989027977,
        "p50_ms_median": 0.10913599655032158,
        "p50_ms_min": 0.10886400192975998,
        "p50_ms_max": 0.10915199667215347,
        "p80_ms_median": 0.11113599687814713,
        "p80_ms_min": 0.110944002866745,
        "p80_ms_max": 0.11123199760913849,
        "tflops_median": 314.83414688165743,
        "tflops_min": 314.78799669787213,
        "tflops_max": 315.62075395840407,
        "relative_to_reference_median": 0.4517736801433816,
        "relative_to_reference_min": 0.4503738756431729,
        "relative_to_reference_max": 0.4541446146553679,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "tall",
        "M": 4096,
        "N": 1024,
        "K": 4096,
        "dtype": "fp16",
        "provider": "fixed manual tl.dot",
        "independent_runs": 3,
        "p20_ms_median": 0.1074879989027977,
        "p20_ms_min": 0.10745599865913391,
        "p20_ms_max": 0.10764800012111664,
        "p50_ms_median": 0.10911999642848969,
        "p50_ms_min": 0.1090880036354065,
        "p50_ms_max": 0.10931199789047241,
        "p80_ms_median": 0.11129599809646606,
        "p80_ms_min": 0.11116799712181091,
        "p80_ms_max": 0.11139199882745743,
        "tflops_median": 314.88031059932445,
        "tflops_min": 314.3272379160749,
        "tflops_max": 314.97265714786556,
        "relative_to_reference_median": 0.4520386989241026,
        "relative_to_reference_min": 0.44964873652567594,
        "relative_to_reference_max": 0.4530791956049221,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "tall",
        "M": 4096,
        "N": 1024,
        "K": 4096,
        "dtype": "fp16",
        "provider": "optimized Triton",
        "independent_runs": 3,
        "p20_ms_median": 0.05347200110554695,
        "p20_ms_min": 0.053408000618219376,
        "p20_ms_max": 0.05350400134921074,
        "p50_ms_median": 0.0541439987719059,
        "p50_ms_min": 0.05407999828457832,
        "p50_ms_max": 0.0541439987719059,
        "p80_ms_median": 0.05484800040721893,
        "p80_ms_min": 0.05484800040721893,
        "p80_ms_max": 0.05488000065088272,
        "tflops_median": 634.5992011552072,
        "tflops_min": 634.5992011552072,
        "tflops_max": 635.350211869333,
        "relative_to_reference_median": 0.9118343342410744,
        "relative_to_reference_min": 0.9078014711401801,
        "relative_to_reference_max": 0.9131205918962413,
        "autotune_configs_across_runs": "BM=128,BN=256,BK=64,GM=8,warps=8,stages=3"
    },
    {
        "case": "tall",
        "M": 4096,
        "N": 1024,
        "K": 4096,
        "dtype": "fp16",
        "provider": "scalar tl.sum",
        "independent_runs": 3,
        "p20_ms_median": 82.6428451538086,
        "p20_ms_min": 82.59318542480469,
        "p20_ms_max": 82.65248107910156,
        "p50_ms_median": 82.64484786987305,
        "p50_ms_min": 82.59480285644531,
        "p50_ms_max": 82.65305709838867,
        "p80_ms_median": 82.64761352539062,
        "p80_ms_min": 82.5968017578125,
        "p80_ms_max": 82.65350341796875,
        "tflops_median": 0.4157517286752165,
        "tflops_min": 0.4157104355752843,
        "tflops_max": 0.4160036367871629,
        "relative_to_reference_median": 0.0005970351344900492,
        "relative_to_reference_min": 0.0005946785692395943,
        "relative_to_reference_max": 0.0005982224116871379,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "wide",
        "M": 1024,
        "N": 4096,
        "K": 4096,
        "dtype": "fp16",
        "provider": "PyTorch reference",
        "independent_runs": 3,
        "p20_ms_median": 0.04902400076389313,
        "p20_ms_min": 0.048928000032901764,
        "p20_ms_max": 0.04915200173854828,
        "p50_ms_median": 0.049375999718904495,
        "p50_ms_min": 0.049247998744249344,
        "p50_ms_max": 0.04950400069355965,
        "p80_ms_median": 0.049984000623226166,
        "p80_ms_min": 0.04979199916124344,
        "p80_ms_max": 0.05008000135421753,
        "tflops_median": 695.8793455040618,
        "tflops_min": 694.0800316462124,
        "tflops_max": 697.6880125918246,
        "relative_to_reference_median": 1,
        "relative_to_reference_min": 1,
        "relative_to_reference_max": 1,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "wide",
        "M": 1024,
        "N": 4096,
        "K": 4096,
        "dtype": "fp16",
        "provider": "fixed block pointer",
        "independent_runs": 3,
        "p20_ms_median": 0.10947199910879135,
        "p20_ms_min": 0.10940799862146378,
        "p20_ms_max": 0.10956799983978271,
        "p50_ms_median": 0.11078400164842606,
        "p50_ms_min": 0.11078400164842606,
        "p50_ms_max": 0.11081600189208984,
        "p80_ms_median": 0.11212799698114395,
        "p80_ms_min": 0.11209599673748016,
        "p80_ms_max": 0.11215999722480774,
        "tflops_median": 310.15072444341666,
        "tflops_min": 310.06116247957357,
        "tflops_max": 310.15072444341666,
        "relative_to_reference_median": 0.445696120235841,
        "relative_to_reference_min": 0.44454070995321404,
        "relative_to_reference_max": 0.4467224935778277,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "wide",
        "M": 1024,
        "N": 4096,
        "K": 4096,
        "dtype": "fp16",
        "provider": "fixed manual tl.dot",
        "independent_runs": 3,
        "p20_ms_median": 0.10886400192975998,
        "p20_ms_min": 0.10873600095510483,
        "p20_ms_max": 0.10911999642848969,
        "p50_ms_median": 0.11030399799346924,
        "p50_ms_min": 0.11020799726247787,
        "p50_ms_max": 0.11046399921178818,
        "p80_ms_median": 0.11161600053310394,
        "p80_ms_min": 0.11155200004577637,
        "p80_ms_max": 0.11184000223875046,
        "tflops_median": 311.5003897685951,
        "tflops_min": 311.0491980479854,
        "tflops_max": 311.7717336443999,
        "relative_to_reference_median": 0.44763563077584817,
        "relative_to_reference_min": 0.44686411120381225,
        "relative_to_reference_max": 0.44814601179382996,
        "autotune_configs_across_runs": ""
    },
    {
        "case": "wide",
        "M": 1024,
        "N": 4096,
        "K": 4096,
        "dtype": "fp16",
        "provider": "optimized Triton",
        "independent_runs": 3,
        "p20_ms_median": 0.05321599915623665,
        "p20_ms_min": 0.05260799825191498,
        "p20_ms_max": 0.05353600159287453,
        "p50_ms_median": 0.05392000079154968,
        "p50_ms_min": 0.05337600037455559,
        "p50_ms_max": 0.05423999950289726,
        "p80_ms_median": 0.05459199845790863,
        "p80_ms_min": 0.0544000007212162,
        "p80_ms_max": 0.05491200089454651,
        "tflops_median": 637.2354945029015,
        "tflops_min": 633.4760081656095,
        "tflops_max": 643.7301057945011,
        "relative_to_reference_median": 0.9133530790297664,
        "relative_to_reference_min": 0.9103244869363808,
        "relative_to_reference_max": 0.9274580400587352,
        "autotune_configs_across_runs": "BM=128,BN=128,BK=32,GM=8,warps=4,stages=4; BM=128,BN=256,BK=64,GM=8,warps=8,stages=3"
    },
    {
        "case": "wide",
        "M": 1024,
        "N": 4096,
        "K": 4096,
        "dtype": "fp16",
        "provider": "scalar tl.sum",
        "independent_runs": 3,
        "p20_ms_median": 82.66496276855469,
        "p20_ms_min": 82.60636901855469,
        "p20_ms_max": 82.70214080810547,
        "p50_ms_median": 82.66737747192383,
        "p50_ms_min": 82.60913467407227,
        "p50_ms_max": 82.70259094238281,
        "p80_ms_median": 82.6693115234375,
        "p80_ms_min": 82.61087799072266,
        "p80_ms_max": 82.70377349853516,
        "tflops_median": 0.4156384225406151,
        "tflops_min": 0.415461450197343,
        "tflops_max": 0.41593146452342833,
        "relative_to_reference_median": 0.0005970308687584376,
        "relative_to_reference_min": 0.0005961568165379457,
        "relative_to_reference_max": 0.0005988335690263381,
        "autotune_configs_across_runs": ""
    }
]


def show_recorded_h200_benchmark(provider=None):
    rows = RECORDED_H200_BENCHMARK_RESULTS
    if provider is not None:
        rows = [row for row in rows if row["provider"] == provider]

    for row in rows:
        print(
            f"{row['case']:14} {row['dtype']:4} "
            f"{row['provider']:24} "
            f"{row['p50_ms_median']:12.6f} ms "
            f"{row['tflops_median']:9.3f} TFLOP/s "
            f"{100 * row['relative_to_reference_median']:7.2f}%"
        )


show_recorded_h200_benchmark()

In [ ]:
RECORDED_H200_CORRECTNESS_RESULTS = [
    {
        "case": "aligned square",
        "dtype": "fp16",
        "provider": "scalar tl.sum",
        "M": 256,
        "N": 256,
        "K": 256,
        "max_abs_error": 0.03125,
        "mean_abs_error": 0.000002762455551419407,
        "max_tolerance_ratio": 0.09057971835136414,
        "status": "pass"
    },
    {
        "case": "aligned square",
        "dtype": "fp16",
        "provider": "optimized Triton",
        "M": 256,
        "N": 256,
        "K": 256,
        "max_abs_error": 0.03125,
        "mean_abs_error": 0.000006414584277081303,
        "max_tolerance_ratio": 0.09057971835136414,
        "status": "pass"
    },
    {
        "case": "aligned square",
        "dtype": "fp16",
        "provider": "fixed manual tl.dot",
        "M": 256,
        "N": 256,
        "K": 256,
        "max_abs_error": 0.03125,
        "mean_abs_error": 0.000006414584277081303,
        "max_tolerance_ratio": 0.09057971835136414,
        "status": "pass"
    },
    {
        "case": "aligned square",
        "dtype": "fp16",
        "provider": "fixed block pointer",
        "M": 256,
        "N": 256,
        "K": 256,
        "max_abs_error": 0.03125,
        "mean_abs_error": 0.000006414584277081303,
        "max_tolerance_ratio": 0.09057971835136414,
        "status": "pass"
    },
    {
        "case": "M remainder",
        "dtype": "fp16",
        "provider": "scalar tl.sum",
        "M": 257,
        "N": 256,
        "K": 256,
        "max_abs_error": 0.015625,
        "mean_abs_error": 0.0000015926195828797063,
        "max_tolerance_ratio": 0.07770008593797684,
        "status": "pass"
    },
    {
        "case": "M remainder",
        "dtype": "fp16",
        "provider": "optimized Triton",
        "M": 257,
        "N": 256,
        "K": 256,
        "max_abs_error": 0.015625,
        "mean_abs_error": 0.000004012318640889134,
        "max_tolerance_ratio": 0.08620689064264297,
        "status": "pass"
    },
    {
        "case": "N remainder",
        "dtype": "fp16",
        "provider": "scalar tl.sum",
        "M": 256,
        "N": 259,
        "K": 256,
        "max_abs_error": 0.015625,
        "mean_abs_error": 0.00000313314876621007,
        "max_tolerance_ratio": 0.08510638773441315,
        "status": "pass"
    },
    {
        "case": "N remainder",
        "dtype": "fp16",
        "provider": "optimized Triton",
        "M": 256,
        "N": 259,
        "K": 256,
        "max_abs_error": 0.03125,
        "mean_abs_error": 0.000005427543783298461,
        "max_tolerance_ratio": 0.08576329797506332,
        "status": "pass"
    },
    {
        "case": "K remainder",
        "dtype": "fp16",
        "provider": "scalar tl.sum",
        "M": 256,
        "N": 256,
        "K": 263,
        "max_abs_error": 0.03125,
        "mean_abs_error": 0.0000019526178220985457,
        "max_tolerance_ratio": 0.08006405085325241,
        "status": "pass"
    },
    {
        "case": "K remainder",
        "dtype": "fp16",
        "provider": "optimized Triton",
        "M": 256,
        "N": 256,
        "K": 263,
        "max_abs_error": 0.03125,
        "mean_abs_error": 0.000004106354936084244,
        "max_tolerance_ratio": 0.08006405085325241,
        "status": "pass"
    },
    {
        "case": "all remainders",
        "dtype": "fp16",
        "provider": "scalar tl.sum",
        "M": 257,
        "N": 259,
        "K": 263,
        "max_abs_error": 0.015625,
        "mean_abs_error": 0.00000170110217823094,
        "max_tolerance_ratio": 0.08849557489156723,
        "status": "pass"
    },
    {
        "case": "all remainders",
        "dtype": "fp16",
        "provider": "optimized Triton",
        "M": 257,
        "N": 259,
        "K": 263,
        "max_abs_error": 0.015625,
        "mean_abs_error": 0.000004544742296275217,
        "max_tolerance_ratio": 0.08849557489156723,
        "status": "pass"
    },
    {
        "case": "small boundary",
        "dtype": "fp16",
        "provider": "scalar tl.sum",
        "M": 31,
        "N": 47,
        "K": 29,
        "max_abs_error": 0.0078125,
        "mean_abs_error": 0.000005362045158108231,
        "max_tolerance_ratio": 0.07352941483259201,
        "status": "pass"
    },
    {
        "case": "small boundary",
        "dtype": "fp16",
        "provider": "optimized Triton",
        "M": 31,
        "N": 47,
        "K": 29,
        "max_abs_error": 0.0078125,
        "mean_abs_error": 0.000005362045158108231,
        "max_tolerance_ratio": 0.07352941483259201,
        "status": "pass"
    },
    {
        "case": "tall",
        "dtype": "fp16",
        "provider": "scalar tl.sum",
        "M": 769,
        "N": 193,
        "K": 257,
        "max_abs_error": 0.015625,
        "mean_abs_error": 0.000003013105015270412,
        "max_tolerance_ratio": 0.08410429209470749,
        "status": "pass"
    },
    {
        "case": "tall",
        "dtype": "fp16",
        "provider": "optimized Triton",
        "M": 769,
        "N": 193,
        "K": 257,
        "max_abs_error": 0.03125,
        "mean_abs_error": 0.000004659818387153791,
        "max_tolerance_ratio": 0.08952552080154419,
        "status": "pass"
    },
    {
        "case": "wide",
        "dtype": "fp16",
        "provider": "scalar tl.sum",
        "M": 193,
        "N": 769,
        "K": 257,
        "max_abs_error": 0.03125,
        "mean_abs_error": 0.0000037729703308286844,
        "max_tolerance_ratio": 0.09082652628421783,
        "status": "pass"
    },
    {
        "case": "wide",
        "dtype": "fp16",
        "provider": "optimized Triton",
        "M": 193,
        "N": 769,
        "K": 257,
        "max_abs_error": 0.03125,
        "mean_abs_error": 0.0000051774204621324316,
        "max_tolerance_ratio": 0.08673027157783508,
        "status": "pass"
    },
    {
        "case": "aligned square",
        "dtype": "bf16",
        "provider": "scalar tl.sum",
        "M": 256,
        "N": 256,
        "K": 256,
        "max_abs_error": 0.0625,
        "mean_abs_error": 0.000001671643985901028,
        "max_tolerance_ratio": 0.16025641560554504,
        "status": "pass"
    },
    {
        "case": "aligned square",
        "dtype": "bf16",
        "provider": "optimized Triton",
        "M": 256,
        "N": 256,
        "K": 256,
        "max_abs_error": 0.125,
        "mean_abs_error": 0.0000064774430938996375,
        "max_tolerance_ratio": 0.2380952537059784,
        "status": "pass"
    },
    {
        "case": "aligned square",
        "dtype": "bf16",
        "provider": "fixed manual tl.dot",
        "M": 256,
        "N": 256,
        "K": 256,
        "max_abs_error": 0.125,
        "mean_abs_error": 0.0000064774430938996375,
        "max_tolerance_ratio": 0.2380952537059784,
        "status": "pass"
    },
    {
        "case": "aligned square",
        "dtype": "bf16",
        "provider": "fixed block pointer",
        "M": 256,
        "N": 256,
        "K": 256,
        "max_abs_error": 0.125,
        "mean_abs_error": 0.0000064774430938996375,
        "max_tolerance_ratio": 0.2380952537059784,
        "status": "pass"
    },
    {
        "case": "M remainder",
        "dtype": "bf16",
        "provider": "scalar tl.sum",
        "M": 257,
        "N": 256,
        "K": 256,
        "max_abs_error": 0.001953125,
        "mean_abs_error": 4.094195560355729e-8,
        "max_tolerance_ratio": 0.043572988361120224,
        "status": "pass"
    },
    {
        "case": "M remainder",
        "dtype": "bf16",
        "provider": "optimized Triton",
        "M": 257,
        "N": 256,
        "K": 256,
        "max_abs_error": 0.125,
        "mean_abs_error": 0.000002717023107834393,
        "max_tolerance_ratio": 0.1862197369337082,
        "status": "pass"
    },
    {
        "case": "N remainder",
        "dtype": "bf16",
        "provider": "scalar tl.sum",
        "M": 256,
        "N": 259,
        "K": 256,
        "max_abs_error": 0.03125,
        "mean_abs_error": 5.007710797144682e-7,
        "max_tolerance_ratio": 0.11947431415319443,
        "status": "pass"
    },
    {
        "case": "N remainder",
        "dtype": "bf16",
        "provider": "optimized Triton",
        "M": 256,
        "N": 259,
        "K": 256,
        "max_abs_error": 0.0625,
        "mean_abs_error": 0.0000017676574088909547,
        "max_tolerance_ratio": 0.20576131343841553,
        "status": "pass"
    },
    {
        "case": "K remainder",
        "dtype": "bf16",
        "provider": "scalar tl.sum",
        "M": 256,
        "N": 256,
        "K": 263,
        "max_abs_error": 0.125,
        "mean_abs_error": 0.00000193924643099308,
        "max_tolerance_ratio": 0.16260163486003876,
        "status": "pass"
    },
    {
        "case": "K remainder",
        "dtype": "bf16",
        "provider": "optimized Triton",
        "M": 256,
        "N": 256,
        "K": 263,
        "max_abs_error": 0.125,
        "mean_abs_error": 0.000002901244442909956,
        "max_tolerance_ratio": 0.173611119389534,
        "status": "pass"
    },
    {
        "case": "all remainders",
        "dtype": "bf16",
        "provider": "scalar tl.sum",
        "M": 257,
        "N": 259,
        "K": 263,
        "max_abs_error": 0.03125,
        "mean_abs_error": 4.7092709110074793e-7,
        "max_tolerance_ratio": 0.13227513432502747,
        "status": "pass"
    },
    {
        "case": "all remainders",
        "dtype": "bf16",
        "provider": "optimized Triton",
        "M": 257,
        "N": 259,
        "K": 263,
        "max_abs_error": 0.125,
        "mean_abs_error": 0.000003057852609344991,
        "max_tolerance_ratio": 0.2298850566148758,
        "status": "pass"
    },
    {
        "case": "small boundary",
        "dtype": "bf16",
        "provider": "scalar tl.sum",
        "M": 31,
        "N": 47,
        "K": 29,
        "max_abs_error": 0,
        "mean_abs_error": 0,
        "max_tolerance_ratio": 0,
        "status": "pass"
    },
    {
        "case": "small boundary",
        "dtype": "bf16",
        "provider": "optimized Triton",
        "M": 31,
        "N": 47,
        "K": 29,
        "max_abs_error": 0,
        "mean_abs_error": 0,
        "max_tolerance_ratio": 0,
        "status": "pass"
    },
    {
        "case": "tall",
        "dtype": "bf16",
        "provider": "scalar tl.sum",
        "M": 769,
        "N": 193,
        "K": 257,
        "max_abs_error": 0.125,
        "mean_abs_error": 0.000001510134779891814,
        "max_tolerance_ratio": 0.21786494553089142,
        "status": "pass"
    },
    {
        "case": "tall",
        "dtype": "bf16",
        "provider": "optimized Triton",
        "M": 769,
        "N": 193,
        "K": 257,
        "max_abs_error": 0.25,
        "mean_abs_error": 0.0000033495387015136657,
        "max_tolerance_ratio": 0.22522523999214172,
        "status": "pass"
    },
    {
        "case": "wide",
        "dtype": "bf16",
        "provider": "scalar tl.sum",
        "M": 193,
        "N": 769,
        "K": 257,
        "max_abs_error": 0.125,
        "mean_abs_error": 0.0000012677788845394389,
        "max_tolerance_ratio": 0.16501650214195251,
        "status": "pass"
    },
    {
        "case": "wide",
        "dtype": "bf16",
        "provider": "optimized Triton",
        "M": 193,
        "N": 769,
        "K": 257,
        "max_abs_error": 0.125,
        "mean_abs_error": 0.0000016252855630227714,
        "max_tolerance_ratio": 0.21231421828269958,
        "status": "pass"
    }
]

recorded_correctness_passes = sum(
    row["status"] == "pass"
    for row in RECORDED_H200_CORRECTNESS_RESULTS
)
recorded_max_tolerance_ratio = max(
    row["max_tolerance_ratio"]
    for row in RECORDED_H200_CORRECTNESS_RESULTS
)
print(
    f"Recorded H200 correctness: {recorded_correctness_passes}/"
    f"{len(RECORDED_H200_CORRECTNESS_RESULTS)} passed; "
    f"largest tolerance ratio = {recorded_max_tolerance_ratio:.4f}"
)

## Optional profiler target

Attach a GPU profiler to the running notebook kernel, then call `run_matmul_profile_target`. The first launch selects and caches the optimized configuration before the repeated launches. The recorded H200 session could not read performance counters because Nsight Compute returned `ERR_NVGPUCTRPERM`, so the chapter does not claim measured HBM traffic or Tensor Core utilization.

In [ ]:
RUN_PROFILE_TARGET = False

if RUN_PROFILE_TARGET:
    profiled_output = run_matmul_profile_target(
        provider="optimized",
        M=4096,
        N=4096,
        K=4096,
        dtype_name="fp16",
        repetitions=10,
    )
    print(profiled_output.shape)

## Roofline revisited

The H200 roof uses the published 4.8 TB/s HBM bandwidth and half of the published sparse FP16 rate as the calculated dense ceiling. The optimized point has a measured vertical coordinate, but its exact horizontal coordinate requires counter-derived HBM bytes. The notebook therefore calculates the 127.9 FLOP/byte lower bound and draws an arrow to the right. Pass measured HBM bytes to `roofline_revisited_values` if a future run has counter access.

In [ ]:
def recorded_h200_result(case_name, provider):
    return next(
        row
        for row in RECORDED_H200_BENCHMARK_RESULTS
        if row["case"] == case_name and row["provider"] == provider
    )


def roofline_revisited_values(profiled_hbm_bytes=None):
    M = N = K = 4096
    hbm_bandwidth_tb_s = 4.8
    dense_fp16_tflops = 1979.0 / 2
    optimized = recorded_h200_result(
        "square 4096",
        "optimized Triton",
    )
    scalar = recorded_h200_result(
        "square 4096",
        "scalar tl.sum",
    )
    optimized_tflops = optimized["tflops_median"]
    values = {
        "hbm_bandwidth_tb_s": hbm_bandwidth_tb_s,
        "dense_fp16_tflops": dense_fp16_tflops,
        "ridge_point_flops_per_byte": (
            dense_fp16_tflops / hbm_bandwidth_tb_s
        ),
        "scalar_source_intensity_flops_per_byte": 0.5,
        "scalar_measured_tflops": scalar["tflops_median"],
        "optimized_measured_tflops": optimized_tflops,
        "optimized_intensity_lower_bound": (
            optimized_tflops / hbm_bandwidth_tb_s
        ),
        "optimized_fraction_of_dense_roof": (
            optimized_tflops / dense_fp16_tflops
        ),
        "optimized_fraction_of_reference": (
            optimized["relative_to_reference_median"]
        ),
        "profiled_intensity_flops_per_byte": None,
    }

    if profiled_hbm_bytes is not None:
        values["profiled_intensity_flops_per_byte"] = (
            2 * M * N * K / profiled_hbm_bytes
        )

    return values


roofline_values = roofline_revisited_values()
for name, value in roofline_values.items():
    print(f"{name}: {value}")

In [ ]:
def plot_roofline_revisited():
    import matplotlib.pyplot as plt

    values = roofline_revisited_values()
    intensities = np.logspace(-1, 3, 500)
    attainable = np.minimum(
        values["hbm_bandwidth_tb_s"] * intensities,
        values["dense_fp16_tflops"],
    )
    figure, axis = plt.subplots(figsize=(7.2, 4.4))
    axis.loglog(
        intensities,
        attainable,
        color="black",
        linewidth=1.8,
        label="Calculated dense FP16 roof",
    )
    axis.scatter(
        [values["scalar_source_intensity_flops_per_byte"]],
        [values["scalar_measured_tflops"]],
        marker="s",
        color="#0072B2",
        label="Scalar source estimate",
        zorder=3,
    )
    lower_bound = values["optimized_intensity_lower_bound"]
    optimized_tflops = values["optimized_measured_tflops"]
    axis.scatter(
        [lower_bound],
        [optimized_tflops],
        marker="o",
        color="#D55E00",
        label="Optimized lower bound",
        zorder=3,
    )
    axis.annotate(
        "",
        xy=(700, optimized_tflops),
        xytext=(lower_bound, optimized_tflops),
        arrowprops={"arrowstyle": "->", "color": "#D55E00"},
    )
    axis.set_xlim(0.1, 1000)
    axis.set_ylim(0.1, 1200)
    axis.set_xlabel("Operational intensity (FLOP/byte)")
    axis.set_ylabel("Throughput (TFLOP/s)")
    axis.grid(True, which="both", color="#D9D9D9", linewidth=0.6)
    axis.legend()
    figure.tight_layout()
    return figure, axis


roofline_figure, roofline_axis = plot_roofline_revisited()